-- Import packages, set fantasy scoring method
-- load stats from 2025 and 2026 for all WRs


In [1]:
import nflreadpy as nfl
import pandas as pd

SCORING = "ppr"   # change to "half" or "ppr"

stats = nfl.load_player_stats([2025, 2026]).to_pandas()

wr_hist = stats[(stats["position"] == "WR") & (stats["season_type"] == "REG")].copy()

wr_hist["fp_std"] = wr_hist["fantasy_points"]
wr_hist["fp_ppr"] = wr_hist["fantasy_points_ppr"]
wr_hist["fp_half"] = (wr_hist["fp_std"] + wr_hist["fp_ppr"]) / 2

fp_col = {"standard": "fp_std", "half": "fp_half", "ppr": "fp_ppr"}[SCORING]
wr_hist["fp"] = wr_hist[fp_col]

print(len(wr_hist), "WR weekly rows loaded")

3064 WR weekly rows loaded


-- Sort all WRs by player id, then season, then week
-- Group by player id to total player's stats
-- Perform functions on stats
-- Create basic comparison metrics
-- Create status comparisons
-- Get current active players


In [2]:
wr_hist = wr_hist.sort_values(["player_id", "season", "week"])
recent = wr_hist.groupby("player_id").tail(8)

fantasy = (
    recent.groupby(["player_id", "player_display_name"])
          .agg(
              games=("week", "count"),
              games_2026=("season", lambda s: (s == 2026).sum()),
              oldest_season=("season", "min"),
              targets=("targets", "sum"),
              target_share=("target_share", "mean"),
              air_yards_share=("air_yards_share", "mean"),
              wopr=("wopr", "mean"),
              fp=("fp", "sum"),
              tds=("receiving_tds", "sum"),
          )
          .reset_index()
)

fantasy["fp_per_game"] = fantasy["fp"] / fantasy["games"]
fantasy["targets_per_game"] = fantasy["targets"] / fantasy["games"]

def status(row):
    if row["games_2026"] == 0:
        return "no 2026 games"
    if row["games"] < 4:
        return "limited history"
    if row["oldest_season"] == 2025:
        return "mixed 2025/2026"
    return "2026 only"

fantasy["window"] = fantasy.apply(status, axis=1)
fantasy["window"].value_counts()


window
mixed 2025/2026    155
no 2026 games       84
limited history     28
2026 only           10
Name: count, dtype: int64

-- Build working data set

In [3]:
current = fantasy[fantasy["window"] != "no 2026 games"].copy()
current = current[current["targets"] >= 5]
current = current.dropna(subset=["targets_per_game", "wopr", "fp_per_game"])
print(len(current))

151


-- Predict FPTS from usage

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = current[["targets_per_game", "wopr"]]
y = current["fp_per_game"]

model = LinearRegression()

# how well does usage predict fantasy points on data the model hasn't seen?
scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print("R² per fold:", scores.round(2))
print("Mean R²:", scores.mean().round(2))

R² per fold: [0.85 0.88 0.84 0.79 0.55]
Mean R²: 0.78


-- Expected vs Actual

-- A positive gap shows higher FPTS scoring than what usage would predict. (Touchdown luck, big plays, etc. These tend to be less consistent)
-- A negative gap shows lower FPTS scoring than what usage would predict. (High potential to increase production.)

In [5]:
model.fit(X, y)
current["expected_fp"] = model.predict(X)
current["gap"] = current["fp_per_game"] - current["expected_fp"]

cols = ["player_display_name", "window", "targets_per_game", "wopr",
        "fp_per_game", "expected_fp", "gap"]

print("Scoring MORE than usage suggests (sell-high candidates):")
display(current.sort_values("gap", ascending=False)[cols].head(10).round(2))

print("Scoring LESS than usage suggests (buy-low candidates):")
display(current.sort_values("gap")[cols].head(10).round(2))

Scoring MORE than usage suggests (sell-high candidates):


,player_display_name,window,targets_per_game,wopr,fp_per_game,expected_fp,gap
165,Zay Flowers,mixed 2025/2026,7.00,0.78,20.95,13.76,7.19
167,Puka Nacua,mixed 2025/2026,11.12,0.69,25.64,20.09,5.55
135,Jaxon Smith-Njigba,mixed 2025/2026,11.00,0.93,25.58,20.82,4.76
128,Christian Watson,mixed 2025/2026,6.62,0.50,16.74,12.03,4.71
85,Nico Collins,mixed 2025/2026,7.00,0.59,17.53,13.00,4.52
116,Alec Pierce,mixed 2025/2026,4.62,0.45,12.42,8.61,3.81
263,Josh Cameron,2026 only,1.50,0.14,6.10,2.33,3.77
239,Tory Horton,mixed 2025/2026,3.12,0.30,9.11,5.61,3.50
178,Jalen Coker,mixed 2025/2026,5.75,0.49,13.95,10.60,3.35
254,Denzel Boston,2026 only,5.50,0.63,13.85,10.74,3.11


Scoring LESS than usage suggests (buy-low candidates):


,player_display_name,window,targets_per_game,wopr,fp_per_game,expected_fp,gap
275,KC Concepcion,2026 only,7.00,0.51,7.58,12.70,-5.13
274,Kevin Coleman Jr.,limited history,3.00,0.24,0.65,5.15,-4.50
193,Adonai Mitchell,mixed 2025/2026,7.50,0.70,9.85,14.26,-4.41
268,Carnell Tate,2026 only,8.00,0.77,11.70,15.34,-3.64
119,Jahan Dotson,mixed 2025/2026,3.50,0.36,2.86,6.46,-3.59
31,Courtland Sutton,mixed 2025/2026,7.50,0.61,10.35,13.91,-3.56
1,Keenan Allen,mixed 2025/2026,6.38,0.43,8.19,11.35,-3.16
143,Xavier Hutchinson,mixed 2025/2026,5.25,0.39,6.41,9.40,-2.99
115,John Metchie III,mixed 2025/2026,4.75,0.32,5.43,8.31,-2.89
248,Makai Lemon,2026 only,3.75,0.31,4.00,6.67,-2.67


-- Measure each player compared to last season

In [6]:
from sklearn.linear_model import LinearRegression

reg25 = wr_hist[wr_hist["season"] == 2025]

base = (
    reg25.groupby("player_id")
         .agg(games=("week", "count"),
              targets=("targets", "sum"),
              wopr=("wopr", "mean"),
              fp=("fp", "sum"))
         .reset_index()
)
base["fp_per_game"] = base["fp"] / base["games"]
base["targets_per_game"] = base["targets"] / base["games"]
base = base[(base["games"] >= 8) & (base["targets"] >= 40)].dropna()

feats = ["targets_per_game", "wopr"]
model = LinearRegression().fit(base[feats], base["fp_per_game"])

base["gap_2025"] = base["fp_per_game"] - model.predict(base[feats])
base.sort_values("gap_2025", ascending=False).head(10)

,player_id,games,targets,wopr,fp,fp_per_game,targets_per_game,gap_2025
165,00-0039075,16,166,0.675010,375.0,23.437500,10.375000,4.645973
127,00-0038124,10,55,0.547865,132.4,13.240000,5.500000,3.359350
141,00-0038608,14,46,0.330421,124.1,8.864286,3.285714,3.345525
134,00-0038543,17,163,0.903673,359.9,21.170588,9.588235,3.326488
107,00-0037247,17,137,0.563192,291.9,17.170588,8.058824,2.713969
77,00-0036410,15,98,0.504925,211.6,14.106667,6.533333,2.474839
106,00-0037240,17,102,0.502919,219.9,12.935294,6.000000,2.254891
115,00-0037664,15,84,0.548185,183.3,12.220000,5.600000,2.161068
239,00-0040735,15,60,0.270987,127.9,8.526667,4.000000,1.856002
6,00-0031588,17,102,0.469028,210.3,12.370588,6.000000,1.756921


-- Adjust this season's expectations based on that gap

In [7]:
SHRINK = 0.25

current = current.drop(columns=["expected_fp", "gap"], errors="ignore")
current = current.merge(base[["player_id", "gap_2025"]], on="player_id", how="left")
current["has_baseline"] = current["gap_2025"].notna()
current["gap_2025"] = current["gap_2025"].fillna(0)

current["expected_fp"] = model.predict(current[feats]) + SHRINK * current["gap_2025"]
current["gap"] = current["fp_per_game"] - current["expected_fp"]

cols = ["player_display_name", "window", "has_baseline", "fp_per_game",
        "expected_fp", "gap_2025", "gap"]

print("Sell-high candidates:")
display(current.sort_values("gap", ascending=False)[cols].head(10).round(2))
print("Buy-low candidates:")
display(current.sort_values("gap")[cols].head(10).round(2))

Sell-high candidates:


,player_display_name,window,has_baseline,fp_per_game,expected_fp,gap_2025,gap
89,Zay Flowers,mixed 2025/2026,True,20.95,13.40,1.57,7.55
39,Nico Collins,mixed 2025/2026,True,17.53,12.78,0.64,4.74
70,Jaxon Smith-Njigba,mixed 2025/2026,True,25.58,21.24,3.33,4.34
91,Puka Nacua,mixed 2025/2026,True,25.64,21.32,4.65,4.32
144,Josh Cameron,2026 only,False,6.10,1.96,0.00,4.14
67,Christian Watson,mixed 2025/2026,True,16.74,12.62,3.36,4.12
129,Tory Horton,mixed 2025/2026,False,9.11,5.18,0.00,3.94
139,Denzel Boston,2026 only,False,13.85,10.05,0.00,3.80
58,Alec Pierce,mixed 2025/2026,True,12.42,8.67,2.16,3.75
99,Jalen Coker,mixed 2025/2026,True,13.95,10.59,1.51,3.36


Buy-low candidates:


,player_display_name,window,has_baseline,fp_per_game,expected_fp,gap_2025,gap
150,KC Concepcion,2026 only,False,7.58,12.47,0.00,-4.90
149,Kevin Coleman Jr.,limited history,False,0.65,4.82,0.00,-4.17
107,Adonai Mitchell,mixed 2025/2026,True,9.85,13.08,-2.62,-3.23
14,Courtland Sutton,mixed 2025/2026,True,10.35,13.52,-0.16,-3.17
60,Jahan Dotson,mixed 2025/2026,False,2.86,5.96,0.00,-3.10
145,Carnell Tate,2026 only,False,11.70,14.77,0.00,-3.07
77,Xavier Hutchinson,mixed 2025/2026,True,6.41,9.19,0.25,-2.78
57,John Metchie III,mixed 2025/2026,True,5.43,7.95,-0.59,-2.53
0,Keenan Allen,mixed 2025/2026,True,8.19,10.69,-2.00,-2.51
24,DK Metcalf,mixed 2025/2026,True,11.75,14.13,0.55,-2.38


-- Split lists and ADD Relevance filter

In [8]:
relevant = current[current["targets_per_game"] >= 4]   # adjust to taste

with_hist = relevant[relevant["has_baseline"]]
no_hist = relevant[~relevant["has_baseline"]]

cols = ["player_display_name", "window", "fp_per_game", "expected_fp", "gap_2025", "gap"]

print("Sell-high (players with a 2025 baseline):")
display(with_hist.sort_values("gap", ascending=False)[cols].head(10).round(2))
print("Buy-low (players with a 2025 baseline):")
display(with_hist.sort_values("gap")[cols].head(10).round(2))

print("No 2025 baseline (rookies and returners), shown separately:")
display(no_hist.sort_values("gap")[cols].round(2))

Sell-high (players with a 2025 baseline):


,player_display_name,window,fp_per_game,expected_fp,gap_2025,gap
89,Zay Flowers,mixed 2025/2026,20.95,13.40,1.57,7.55
39,Nico Collins,mixed 2025/2026,17.53,12.78,0.64,4.74
70,Jaxon Smith-Njigba,mixed 2025/2026,25.58,21.24,3.33,4.34
91,Puka Nacua,mixed 2025/2026,25.64,21.32,4.65,4.32
67,Christian Watson,mixed 2025/2026,16.74,12.62,3.36,4.12
58,Alec Pierce,mixed 2025/2026,12.42,8.67,2.16,3.75
99,Jalen Coker,mixed 2025/2026,13.95,10.59,1.51,3.36
51,Chris Olave,mixed 2025/2026,21.80,18.96,-1.03,2.84
44,Ja'Marr Chase,mixed 2025/2026,18.99,16.35,-1.42,2.64
32,Jauan Jennings,mixed 2025/2026,10.82,8.25,0.92,2.57


Buy-low (players with a 2025 baseline):


,player_display_name,window,fp_per_game,expected_fp,gap_2025,gap
107,Adonai Mitchell,mixed 2025/2026,9.85,13.08,-2.62,-3.23
14,Courtland Sutton,mixed 2025/2026,10.35,13.52,-0.16,-3.17
77,Xavier Hutchinson,mixed 2025/2026,6.41,9.19,0.25,-2.78
57,John Metchie III,mixed 2025/2026,5.43,7.95,-0.59,-2.53
0,Keenan Allen,mixed 2025/2026,8.19,10.69,-2.00,-2.51
24,DK Metcalf,mixed 2025/2026,11.75,14.13,0.55,-2.38
31,Michael Pittman,mixed 2025/2026,6.54,8.91,0.38,-2.37
10,Mack Hollins,mixed 2025/2026,8.52,10.44,0.06,-1.92
40,Jaylen Waddle,mixed 2025/2026,9.00,10.84,0.72,-1.84
87,Josh Downs,mixed 2025/2026,9.29,11.09,-0.90,-1.80


No 2025 baseline (rookies and returners), shown separately:


,player_display_name,window,fp_per_game,expected_fp,gap_2025,gap
150,KC Concepcion,2026 only,7.58,12.47,0.0,-4.90
145,Carnell Tate,2026 only,11.70,14.77,0.0,-3.07
93,Malik Nabers,mixed 2025/2026,12.74,13.81,0.0,-1.07
49,Kevin Austin Jr.,mixed 2025/2026,5.78,6.65,0.0,-0.87
59,Garrett Wilson,mixed 2025/2026,13.32,13.95,0.0,-0.62
140,Antonio Williams,2026 only,7.60,8.13,0.0,-0.53
137,Dohnte Meyers,2026 only,6.22,6.48,0.0,-0.25
142,Ted Hurst III,2026 only,6.70,6.87,0.0,-0.17
97,Devaughn Vele,mixed 2025/2026,11.20,11.27,0.0,-0.07
100,Roman Wilson,mixed 2025/2026,7.38,7.05,0.0,0.33


In [9]:
stats_all = nfl.load_player_stats(list(range(2021, 2027))).to_pandas()

wr_hist = stats_all[(stats_all["position"] == "WR") & (stats_all["season_type"] == "REG")].copy()
wr_hist["fp"] = wr_hist["fantasy_points"]     # standard scoring, as before

In [10]:
from sklearn.linear_model import LinearRegression

bt = wr_hist.sort_values(["player_id", "season", "week"]).copy()
bt["t"] = bt["season"] * 100 + bt["week"]      # orders games in time

g = bt.groupby("player_id")
for col in ["fp", "targets", "wopr"]:
    # shift(1) excludes the current game, so each row only sees past games
    bt[col + "_r8"] = g[col].transform(lambda s: s.shift(1).rolling(8, min_periods=4).mean())

bt = bt.dropna(subset=["fp", "fp_r8", "targets_r8", "wopr_r8"])
bt = bt[bt["targets_r8"] >= 4]                  # relevance filter, same idea as before
print(len(bt), "player-games to test on")

5693 player-games to test on


In [11]:
parts = []
for t in sorted(bt["t"].unique()):
    train = bt[bt["t"] < t]                      # only earlier games
    test = bt[bt["t"] == t].copy()
    if len(train) < 200 or test.empty:
        continue
    m = LinearRegression().fit(train[["targets_r8", "wopr_r8"]], train["fp_r8"])
    test["exp_r8"] = m.predict(test[["targets_r8", "wopr_r8"]])
    parts.append(test)

res = pd.concat(parts)
res["gap_r8"] = res["fp_r8"] - res["exp_r8"]
print(len(res), "tested player-games across", res["t"].nunique(), "weeks")

5469 tested player-games across 86 weeks


In [12]:
for lam in [0, 0.25, 0.5, 0.75, 1.0]:
    pred = res["fp_r8"] - lam * res["gap_r8"]
    mae = (res["fp"] - pred).abs().mean()
    print(f"lam={lam:<4} mean abs error: {mae:.3f}")

lam=0    mean abs error: 4.698
lam=0.25 mean abs error: 4.660
lam=0.5  mean abs error: 4.652
lam=0.75 mean abs error: 4.671
lam=1.0  mean abs error: 4.721


In [13]:
res["change"] = res["fp"] - res["fp_r8"]         # next game minus recent average
res["decile"] = res.groupby("t")["gap_r8"].transform(
    lambda s: pd.qcut(s, 10, labels=False, duplicates="drop")
)

summary = res.groupby("decile")["change"].agg(["mean", "count", "std"])
summary["sem"] = summary["std"] / summary["count"] ** 0.5
summary.round(2)

,mean,count,std,sem
decile,,,,
0,1.13,586,5.73,0.24
1,1.07,545,5.58,0.24
2,0.62,537,5.42,0.23
3,-0.23,539,5.04,0.22
4,-0.12,554,5.94,0.25
5,-0.29,524,5.75,0.25
6,-1.49,529,5.58,0.24
7,-0.75,547,6.29,0.27
8,-1.56,535,6.57,0.28


In [14]:
res["err0"] = (res["fp"] - res["fp_r8"]).abs()
res["err25"] = (res["fp"] - (res["fp_r8"] - 0.25 * res["gap_r8"])).abs()
res["improve"] = res["err0"] - res["err25"]        # positive = adjustment helped

by_week = res.groupby("t")["improve"].mean()
se = by_week.std() / len(by_week) ** 0.5

print("Weeks where the adjustment helped:", (by_week > 0).sum(), "of", len(by_week))
print(f"Average improvement per game: {by_week.mean():.3f} ± {se:.3f} (1 SE)")

# does it hold in both halves of the sample?
half = len(by_week) // 2
print("First half:", by_week.iloc[:half].mean().round(3))
print("Second half:", by_week.iloc[half:].mean().round(3))

Weeks where the adjustment helped: 68 of 86
Average improvement per game: 0.037 ± 0.006 (1 SE)
First half: 0.032
Second half: 0.042


In [15]:
res = res.sort_values(["player_id", "t"])
res["own_gap"] = res.groupby("player_id")["gap_r8"].transform(
    lambda s: s.shift(8).expanding(min_periods=1).mean()   # older, non-overlapping windows
)
sub = res.dropna(subset=["own_gap"]).copy()
sub["gap_adj"] = sub["gap_r8"] - sub["own_gap"]
print(len(sub), "player-games have a long enough history")

for lam in [0, 0.25, 0.5, 0.75, 1.0]:
    for label, col in [("plain gap", "gap_r8"), ("player-adjusted", "gap_adj")]:
        pred = sub["fp_r8"] - lam * sub[col]
        print(f"lam={lam:<4} {label:<16} MAE {(sub['fp'] - pred).abs().mean():.3f}")

4069 player-games have a long enough history
lam=0    plain gap        MAE 4.800
lam=0    player-adjusted  MAE 4.800
lam=0.25 plain gap        MAE 4.762
lam=0.25 player-adjusted  MAE 4.767
lam=0.5  plain gap        MAE 4.755
lam=0.5  player-adjusted  MAE 4.772
lam=0.75 plain gap        MAE 4.775
lam=0.75 player-adjusted  MAE 4.811
lam=1.0  plain gap        MAE 4.828
lam=1.0  player-adjusted  MAE 4.882


In [16]:
LAM = 0.5      # update once the multi-season test settles it
feats = ["targets_per_game", "wopr"]

tool = current[current["targets_per_game"] >= 4].copy()

m = LinearRegression().fit(tool[feats], tool["fp_per_game"])
tool["expected_fp"] = m.predict(tool[feats])
tool["gap"] = tool["fp_per_game"] - tool["expected_fp"]
tool["adj_fp"] = tool["fp_per_game"] - LAM * tool["gap"]

lo, hi = tool["gap"].quantile([0.10, 0.90])
tool["flag"] = "hold"
tool.loc[tool["gap"] >= hi, "flag"] = "sell-high"
tool.loc[tool["gap"] <= lo, "flag"] = "buy-low"

def context(r):
    if not r["has_baseline"]:
        return "no 2025 baseline"
    if r["flag"] == "sell-high" and r["gap_2025"] >= 1.0:
        return "overperformed in 2025 too"
    if r["flag"] == "buy-low" and r["gap_2025"] <= -1.0:
        return "underperformed in 2025 too"
    return ""

tool["context"] = tool.apply(context, axis=1)

show = ["player_display_name", "flag", "fp_per_game", "expected_fp", "adj_fp",
        "gap", "gap_2025", "context", "window"]

view = tool[tool["flag"] != "hold"].sort_values("gap", ascending=False)[show].round(2)
view

,player_display_name,flag,fp_per_game,expected_fp,adj_fp,gap,gap_2025,context,window
89,Zay Flowers,sell-high,20.95,14.12,17.53,6.83,1.57,overperformed in 2025 too,mixed 2025/2026
67,Christian Watson,sell-high,16.74,12.06,14.40,4.68,3.36,overperformed in 2025 too,mixed 2025/2026
91,Puka Nacua,sell-high,25.64,20.99,23.31,4.65,4.65,overperformed in 2025 too,mixed 2025/2026
39,Nico Collins,sell-high,17.53,13.17,15.35,4.35,0.64,,mixed 2025/2026
58,Alec Pierce,sell-high,12.42,8.29,10.36,4.13,2.16,overperformed in 2025 too,mixed 2025/2026
70,Jaxon Smith-Njigba,sell-high,25.58,21.93,23.76,3.65,3.33,overperformed in 2025 too,mixed 2025/2026
99,Jalen Coker,sell-high,13.95,10.49,12.22,3.46,1.51,overperformed in 2025 too,mixed 2025/2026
139,Denzel Boston,sell-high,13.85,10.73,12.29,3.12,0.00,no 2025 baseline,2026 only
66,Wan'Dale Robinson,buy-low,11.84,14.53,13.18,-2.69,-2.24,underperformed in 2025 too,mixed 2025/2026
77,Xavier Hutchinson,buy-low,6.41,9.12,7.77,-2.71,0.25,,mixed 2025/2026
